# Hakka ↔ Mandarin Translation — End-to-End Pipeline

Fine-tune [`yentinglin/Llama-3-Taiwan-8B-Instruct`](https://huggingface.co/yentinglin/Llama-3-Taiwan-8B-Instruct) for **bidirectional Hakka ↔ Mandarin translation** using LoRA via [Unsloth](https://github.com/unslothai/unsloth).

| Stage | Description |
|-------|-------------|
| **1. Data Extraction** | Parse 3 data sources → ~25 k sentence pairs + ~200 k word pairs |
| **2. Fine-tuning** | LoRA fine-tune on extracted data (T4 16 GB, ~20 min) |
| **3. Evaluation** | Sentence BLEU · Corpus BLEU · BERTScore (bert-base-chinese) |

**Data sources**
- `HAKKA_UTF8_TSV/` — 294 sentence pairs, medical dialogue (四縣 dialect)
- `dict-hakka.json` — ~12 k example sentences from the official Hakka dictionary
- `HAT-Lexicon-Sixian/Hailu.csv` — ~100 k word-level pairs (四縣 + 海陸)

**Training formats**
- Hakka → Mandarin (`h2m`)
- Mandarin → Hakka (`m2h`)
- Hakka dialogue (doctor–patient, Hakka only)

---
**Required GPU**: T4 (16 GB) or better.

## 1. Installation

In [1]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, '0.0.34')
    !pip install sentencepiece protobuf "datasets>=2.19" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2
!pip install bert-score -q

## 2. Configuration

In [2]:
import os

# ── Model ──────────────────────────────────────────────────────────────────
MODEL_NAME     = "yentinglin/Llama-3-Taiwan-8B-Instruct"
MAX_SEQ_LENGTH = 2048
LOAD_IN_4BIT   = True

# ── Data paths (adjust for your environment) ───────────────────────────────
DATASET_ROOT = "/content/dataset"          # Colab: unzipped dataset folder
# DATASET_ROOT = "../../dataset"           # Local

TSV_DIR        = os.path.join(DATASET_ROOT, "HAKKA_UTF8_TSV")
DICT_PATH      = os.path.join(DATASET_ROOT, "dict-hakka.json")
LEXICON_SIXIAN = os.path.join(DATASET_ROOT, "hat-lexicon-sixian-master", "HAT-Lexicon-Sixian.csv")
LEXICON_HAILU  = os.path.join(DATASET_ROOT, "hat-lexicon-hailu-master",  "HAT-Lexicon-Hailu.csv")

# ── LoRA ───────────────────────────────────────────────────────────────────
LORA_R     = 16
LORA_ALPHA = 16

# ── Training ───────────────────────────────────────────────────────────────
BATCH_SIZE        = 2
GRAD_ACCUMULATION = 4
NUM_EPOCHS        = 3
LEARNING_RATE     = 2e-4
OUTPUT_DIR        = "outputs/hakka_translation"
TRAIN_RATIO       = 0.90    # 90% train, 10% val (sentence pairs)
INCLUDE_LEXICON   = True    # Set False to skip word-level HAT Lexicon pairs

# ── System prompts — used for BOTH training and evaluation ─────────────────
SYSTEM = {
    "h2m": (
        "你是專業翻譯助手。請將使用者輸入的客語漢字翻成自然、流暢、忠實原意的繁體中文。"
        "只輸出翻譯結果，不要加解釋、不要加前後綴。"
    ),
    "m2h_sixian": (
        "你是專業翻譯助手。請將使用者輸入的繁體中文翻成客語漢字（四縣腔）。"
        "只輸出翻譯結果，不要加解釋、不要加前後綴。"
    ),
    "m2h_hailu": (
        "你是專業翻譯助手。請將使用者輸入的繁體中文翻成客語漢字（海陸腔）。"
        "只輸出翻譯結果，不要加解釋、不要加前後綴。"
    ),
    "dialogue": (
        "你是一位精通客家語（四縣腔）的醫師。"
        "請使用客家語與病人進行問診，詢問必要的症狀細節。"
    ),
}

print(f"Model     : {MODEL_NAME}")
print(f"4-bit     : {LOAD_IN_4BIT}")
print(f"Lexicon   : {INCLUDE_LEXICON}")
print(f"Data root : {DATASET_ROOT}")

Model     : yentinglin/Llama-3-Taiwan-8B-Instruct
4-bit     : True
Lexicon   : True
Data root : /content/dataset


## 3. Mount Drive & Upload Dataset

In [7]:
from google.colab import drive
drive.mount("/content/drive")

import shutil
# Adjust path to wherever you stored the dataset zip on Drive
zip_src = "/content/drive/MyDrive/dataset.zip"
if os.path.exists(zip_src):
    !unzip -q {zip_src} -d /content/
    print("Dataset extracted:", os.listdir(DATASET_ROOT))
else:
    print(f"ZIP not found at {zip_src} — upload manually or adjust zip_src.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Dataset extracted: ['HAKKA_UTF8_TSV', 'hat-lexicon-sixian-master', 'dict-hakka.json', 'hat-lexicon-hailu-master']


## 4. Data Extraction

Three sources → combined training set:

| Source | Type | Pairs |
|--------|------|-------|
| `HAKKA_UTF8_TSV/` | Sentence (medical) | ~294 |
| `dict-hakka.json` | Sentence (general) | ~12 k |
| HAT Lexicon × 2 | Word-level | ~100 k |

Each example stores raw `hakka` / `mandarin` fields for later BLEU evaluation.

In [8]:
import csv, glob, json, random, re
from collections import Counter

# Unicode interlinear annotation markers used inside dict-hakka examples
_FFF9 = "\ufff9"   # start of Hakka annotation
_FFFB = "\ufffb"   # terminator (Mandarin follows)
MIN_LEN = 4


def _make(system_key, human, gpt, hakka, mandarin, direction, source):
    """Create one ShareGPT training example with metadata for evaluation."""
    return {
        "system":        SYSTEM[system_key],
        "conversations": [
            {"from": "human", "value": human},
            {"from": "gpt",   "value": gpt},
        ],
        "hakka":     hakka,
        "mandarin":  mandarin,
        "direction": direction,   # h2m | m2h | dialogue
        "source":    source,      # tsv | dict | hat_sixian | hat_hailu
    }


def _clean(text, min_len=MIN_LEN):
    """Reject too-short or heavily latinised strings."""
    if not text or len(text) < min_len:
        return False
    latin = sum(1 for c in text if c.isascii() and (c.isalpha() or c.isdigit()))
    return (latin / max(len(text), 1)) < 0.25

print("Utility functions ready.")

Utility functions ready.


In [9]:
# ── Source 1: HAKKA_UTF8_TSV ── sentence pairs + Hakka dialogue ──────────────
sentence_pairs_tsv = []
dialogue_pairs     = []

for filepath in sorted(glob.glob(os.path.join(TSV_DIR, "*.txt"))):
    with open(filepath, "r", encoding="utf-16", errors="replace") as f:
        lines = [ln.strip() for ln in f if ln.strip()]

    rows = []
    for line in lines[1:]:                    # skip header
        cols = line.split("\t")
        if len(cols) >= 6:
            rows.append({"speaker": cols[1].strip(),
                          "hakka":   cols[4].strip(),
                          "mandarin":cols[5].strip()})

    # Sentence pairs — both directions
    for r in rows:
        h, m = r["hakka"], r["mandarin"]
        if not _clean(h) or not _clean(m):
            continue
        sentence_pairs_tsv.append(_make("h2m",        h, m, h, m, "h2m", "tsv"))
        sentence_pairs_tsv.append(_make("m2h_sixian", m, h, h, m, "m2h", "tsv"))

    # Dialogue pairs — context-accumulating, all-Hakka doctor role
    history = []
    for r in rows:
        text = r["hakka"]
        if not text:
            continue
        if r["speaker"] == "speaker02":       # patient → human
            history.append({"from": "human", "value": text})
        elif r["speaker"] == "speaker01":     # doctor  → gpt
            if history:
                dialogue_pairs.append({
                    "system":        SYSTEM["dialogue"],
                    "conversations": history.copy() + [{"from": "gpt", "value": text}],
                    "direction":     "dialogue",
                    "source":        "tsv",
                })
            history.append({"from": "gpt", "value": text})

n_sent = len(sentence_pairs_tsv) // 2
print(f"TSV  → {n_sent} sentence pairs ({n_sent*2} examples) | {len(dialogue_pairs)} dialogue examples")

TSV  → 293 sentence pairs (586 examples) | 134 dialogue examples


In [10]:
# ── Source 2: dict-hakka.json ── example sentences ￹Hakka￻Mandarin ───────────
sentence_pairs_dict = []
skipped = Counter()
seen    = set()

with open(DICT_PATH, "r", encoding="utf-8") as f:
    dict_data = json.load(f)

for entry in dict_data:
    title = entry.get("title", "")
    for heteronym in entry.get("heteronyms", []):
        for defn in heteronym.get("definitions", []):
            for ex in defn.get("example", []):
                if _FFF9 not in ex or _FFFB not in ex:
                    continue
                start = ex.index(_FFF9) + 1
                end   = ex.index(_FFFB)
                hakka = ex[start:end].strip()
                mand  = ex[end + 1:].strip()

                # Replace □ placeholder with dictionary headword
                if "□" in hakka and title and "□" not in title:
                    hakka = hakka.replace("□", title)

                if "□" in hakka:
                    skipped["square_left"] += 1; continue
                if not mand:
                    skipped["no_mandarin"] += 1; continue
                if not _clean(hakka) or not _clean(mand):
                    skipped["too_short_noisy"] += 1; continue

                key = hakka[:60]
                if key in seen:
                    continue
                seen.add(key)

                sentence_pairs_dict.append(_make("h2m",        hakka, mand,  hakka, mand, "h2m", "dict"))
                sentence_pairs_dict.append(_make("m2h_sixian", mand,  hakka, hakka, mand, "m2h", "dict"))

n_dict = len(sentence_pairs_dict) // 2
print(f"DICT → {n_dict} sentence pairs ({len(sentence_pairs_dict)} examples) | skipped: {dict(skipped)}")

DICT → 12079 sentence pairs (24158 examples) | skipped: {'square_left': 11, 'no_mandarin': 7585, 'too_short_noisy': 503}


In [15]:
# ── Source 3: HAT Lexicon CSV ── word-level pairs (四縣 + 海陸) ───────────────
# Dedup key = (source, mand, hakka) → giữ tất cả biến thể Hakka per Mandarin word
lexicon_pairs = []

lex_sources = [
    (LEXICON_SIXIAN, "m2h_sixian", "hat_sixian"),
    (LEXICON_HAILU,  "m2h_hailu",  "hat_hailu"),
]

if INCLUDE_LEXICON:
    for lex_path, sys_m2h_key, source_tag in lex_sources:
        if not os.path.exists(lex_path):
            print(f"  Not found: {lex_path}")
            continue
        with open(lex_path, "r", encoding="utf-8") as f:
            rows = list(csv.reader(f))
        seen_lex = set()
        count = 0
        for row in rows[1:]:             # skip header
            if len(row) < 3:
                continue
            mand  = row[0].strip().lstrip("\ufeff")   # strip BOM
            hakka = row[2].strip()
            if not mand or not hakka or mand == hakka:
                continue
            # Keep all Hakka variants per Mandarin word (序號 0, 1, 2…)
            key = f"{source_tag}:{mand}:{hakka}"
            if key in seen_lex:
                continue
            seen_lex.add(key)
            lexicon_pairs.append(_make("h2m",        hakka, mand,  hakka, mand, "h2m", source_tag))
            lexicon_pairs.append(_make(sys_m2h_key,  mand,  hakka, hakka, mand, "m2h", source_tag))
            count += 1
        print(f"  {os.path.basename(lex_path)}: {count} word pairs ({count*2} examples)")
else:
    print("Lexicon skipped (INCLUDE_LEXICON=False).")

  HAT-Lexicon-Sixian.csv: 15552 word pairs (31104 examples)
  HAT-Lexicon-Hailu.csv: 13904 word pairs (27808 examples)


In [16]:
# ── Combine & split ───────────────────────────────────────────────────────────
# Sentence-level: split 90/10 → val set used for BLEU evaluation
all_sentence = sentence_pairs_tsv + sentence_pairs_dict
random.seed(42)
random.shuffle(all_sentence)
split_idx   = max(1, int(len(all_sentence) * TRAIN_RATIO))
sent_train  = all_sentence[:split_idx]
sent_val    = all_sentence[split_idx:]

# Training set = sentence pairs + dialogue + lexicon (all shuffled)
all_train = sent_train + dialogue_pairs + lexicon_pairs
random.shuffle(all_train)

print("Training set breakdown")
print(f"  sentence (train) : {len(sent_train):>7,}")
print(f"  dialogue         : {len(dialogue_pairs):>7,}")
print(f"  lexicon          : {len(lexicon_pairs):>7,}")
print(f"  ─────────────────────────────")
print(f"  total train      : {len(all_train):>7,}")
print(f"  val (sent only)  : {len(sent_val):>7,}")

Training set breakdown
  sentence (train) :  22,269
  dialogue         :     134
  lexicon          :  58,912
  ─────────────────────────────
  total train      :  81,315
  val (sent only)  :   2,475


In [17]:
# ── Sample preview ───────────────────────────────────────────────────────────
from collections import Counter as C

type_counts = C((ex.get("direction"), ex.get("source")) for ex in all_train)
print("Train examples by (direction, source):")
for k, v in sorted(type_counts.items()):
    print(f"  {str(k):<30} {v:>7,}")

print("\n=== Sample — H→M sentence pair ===")
ex = next(e for e in all_train if e["direction"]=="h2m" and e["source"]=="tsv")
print(f"system: {ex['system'][:70]}…")
for t in ex["conversations"]:
    print(f"  [{t['from'].upper()}]: {t['value'][:90]}")

print("\n=== Sample — M→H sentence pair ===")
ex = next(e for e in all_train if e["direction"]=="m2h" and e["source"]=="dict")
print(f"system: {ex['system'][:70]}…")
for t in ex["conversations"]:
    print(f"  [{t['from'].upper()}]: {t['value'][:90]}")

print("\n=== Sample — Hakka dialogue ===")
ex = next(e for e in all_train if e["direction"]=="dialogue")
print(f"system: {ex['system'][:70]}…")
for t in ex["conversations"][:3]:
    print(f"  [{t['from'].upper()}]: {t['value'][:90]}")

Train examples by (direction, source):
  ('dialogue', 'tsv')                134
  ('h2m', 'dict')                 10,848
  ('h2m', 'hat_hailu')            13,904
  ('h2m', 'hat_sixian')           15,552
  ('h2m', 'tsv')                     273
  ('m2h', 'dict')                 10,874
  ('m2h', 'hat_hailu')            13,904
  ('m2h', 'hat_sixian')           15,552
  ('m2h', 'tsv')                     274

=== Sample — H→M sentence pair ===
system: 你是專業翻譯助手。請將使用者輸入的客語漢字翻成自然、流暢、忠實原意的繁體中文。只輸出翻譯結果，不要加解釋、不要加前後綴。…
  [HUMAN]: 你恁久敢有跌倒、受傷、扐重抑係坐久無？
  [GPT]: 你最近有跌倒、受傷、搬重物或久坐嗎？

=== Sample — M→H sentence pair ===
system: 你是專業翻譯助手。請將使用者輸入的繁體中文翻成客語漢字（四縣腔）。只輸出翻譯結果，不要加解釋、不要加前後綴。…
  [HUMAN]: 西洋有一位哥倫布，他發現了新大陸。
  [GPT]: 西洋有一個哥倫布，佢發現了新个大陸。

=== Sample — Hakka dialogue ===
system: 你是一位精通客家語（四縣腔）的醫師。請使用客家語與病人進行問診，詢問必要的症狀細節。…
  [HUMAN]: 劉阿伯你好，今晡日麼个所在無爽快呢？
  [GPT]: 先生，𠊎這幾日試著右片个肩頭緊來緊擎毋會高
  [GPT]: 著衫、梳頭那毛都當艱苦，連擎到耳公脣恁高都無麼个才調。


## 5. Load Model & Tokenizer

In [14]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = MODEL_NAME,
    max_seq_length = MAX_SEQ_LENGTH,
    dtype          = None,           # auto-detect (bf16 on Ampere+, fp16 on T4)
    load_in_4bit   = LOAD_IN_4BIT,
)
print("Model loaded.")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.4: Fast Llama patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/194 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

Model loaded.


## 6. Add LoRA Adapters

In [18]:
model = FastLanguageModel.get_peft_model(
    model,
    r                          = LORA_R,
    lora_alpha                 = LORA_ALPHA,
    target_modules             = ["q_proj", "k_proj", "v_proj", "o_proj",
                                   "gate_proj", "up_proj", "down_proj"],
    lora_dropout               = 0,
    bias                       = "none",
    use_gradient_checkpointing = "unsloth",
    random_state               = 3407,
)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable params: {trainable:,}")

Unsloth 2026.4.4 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


Trainable params: 41,943,040


## 7. Training

In [19]:
from unsloth.chat_templates import get_chat_template
from datasets import Dataset

tokenizer = get_chat_template(
    tokenizer,
    chat_template = "llama-3",
    mapping = {"role": "from", "content": "value", "user": "human", "assistant": "gpt"},
)

def format_examples(batch):
    texts = []
    for convs, sys in zip(batch["conversations"], batch["system"]):
        messages = [{"from": "system", "value": sys}] + convs
        texts.append(tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=False
        ))
    return {"text": texts}

train_dataset = Dataset.from_list(all_train).map(format_examples, batched=True)
val_dataset   = Dataset.from_list(sent_val).map(format_examples,  batched=True)

print(f"Train: {len(train_dataset):,} | Val: {len(val_dataset):,}")
print("\n=== Sample formatted text ===")
print(train_dataset[0]["text"][:400], "...")

Map:   0%|          | 0/81315 [00:00<?, ? examples/s]

Map:   0%|          | 0/2475 [00:00<?, ? examples/s]

Train: 81,315 | Val: 2,475

=== Sample formatted text ===
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

你是專業翻譯助手。請將使用者輸入的客語漢字翻成自然、流暢、忠實原意的繁體中文。只輸出翻譯結果，不要加解釋、不要加前後綴。<|eot_id|><|start_header_id|>user<|end_header_id|>

樹林幼兒園彭屋分班<|eot_id|><|start_header_id|>assistant<|end_header_id|>

樹林幼兒園彭厝分班<|eot_id|> ...


In [ ]:
from trl import SFTConfig, SFTTrainer

trainer = SFTTrainer(
    model              = model,
    tokenizer          = tokenizer,
    train_dataset      = train_dataset,
    eval_dataset       = val_dataset,
    dataset_text_field = "text",
    max_seq_length     = MAX_SEQ_LENGTH,
    packing            = False,
    args = SFTConfig(
        per_device_train_batch_size = BATCH_SIZE,
        gradient_accumulation_steps = GRAD_ACCUMULATION,
        warmup_steps                = 5,
        num_train_epochs            = NUM_EPOCHS,
        learning_rate               = LEARNING_RATE,
        fp16                        = not torch.cuda.is_bf16_supported(),
        bf16                        = torch.cuda.is_bf16_supported(),
        logging_steps               = 10,
        eval_strategy               = "epoch",
        save_strategy               = "epoch",
        save_total_limit            = 2,
        load_best_model_at_end      = True,
        optim                       = "adamw_8bit",
        weight_decay                = 0.01,
        lr_scheduler_type           = "cosine",
        seed                        = 3407,
        output_dir                  = OUTPUT_DIR,
        report_to                   = "none",
    ),
)

gpu = torch.cuda.get_device_properties(0)
print(f"GPU: {gpu.name} | VRAM: {gpu.total_memory/1024**3:.1f} GB")
print(f"Memory before: {torch.cuda.max_memory_reserved()/1024**3:.2f} GB")

stats = trainer.train()
print(f"\nTime : {stats.metrics['train_runtime']/60:.1f} min")
print(f"Loss : {stats.metrics.get('train_loss', 'N/A'):.4f}")
print(f"VRAM : {torch.cuda.max_memory_reserved()/1024**3:.2f} GB")

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/81315 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/2475 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.
GPU: Tesla T4 | VRAM: 14.6 GB
Memory before: 6.71 GB


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 81,315 | Num Epochs = 3 | Total steps = 30,495
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 8,072,220,672 (0.52% trained)


Epoch,Training Loss,Validation Loss


Unsloth: Will smartly offload gradients to save VRAM!


### 7b. Multi-GPU Training (Alternative)

> **Run this cell instead of the single-GPU cell above when you have 2+ GPUs.**
>
> Set `TRAIN_GPU_IDS` to a comma-separated list of GPU indices, e.g. `"0"`, `"1"`, `"0,1"`, `"0,1,2,3"`.
>
> Uses `accelerate notebook_launcher` with DDP. Gradient accumulation steps are automatically divided by the number of GPUs to keep the effective batch size constant.

In [ ]:
# ── GPU selection ──────────────────────────────────────────────────────────────
TRAIN_GPU_IDS = "0"      # <-- change to "0,1" / "0,1,2,3" etc.

import os, torch
os.environ["CUDA_VISIBLE_DEVICES"] = TRAIN_GPU_IDS
N_GPUS = len(TRAIN_GPU_IDS.split(","))

for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {p.name}  {p.total_memory/1024**3:.1f} GB")
print(f"N_GPUS={N_GPUS}  |  eff. batch = {BATCH_SIZE} x {max(1, GRAD_ACCUMULATION // N_GPUS)} x {N_GPUS} = {BATCH_SIZE * GRAD_ACCUMULATION}")

# ── Snapshot data lists so worker processes can pickle them ────────────────────
_train_snapshot = list(all_train)
_val_snapshot   = list(sent_val)

# ── Training function (runs on each GPU process) ───────────────────────────────
def train_fn():
    from unsloth import FastLanguageModel
    from unsloth.chat_templates import get_chat_template
    from trl import SFTConfig, SFTTrainer
    from datasets import Dataset
    import torch, os

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name     = MODEL_NAME,
        max_seq_length = MAX_SEQ_LENGTH,
        dtype          = None,
        load_in_4bit   = LOAD_IN_4BIT,
        # do NOT pass device_map with DDP
    )
    model = FastLanguageModel.get_peft_model(
        model,
        r                          = LORA_R,
        lora_alpha                 = LORA_ALPHA,
        target_modules             = ["q_proj", "k_proj", "v_proj", "o_proj",
                                       "gate_proj", "up_proj", "down_proj"],
        lora_dropout               = 0,
        bias                       = "none",
        use_gradient_checkpointing = "unsloth",
        random_state               = 3407,
    )
    tokenizer = get_chat_template(tokenizer, chat_template="llama-3",
        mapping={"role": "from", "content": "value", "user": "human", "assistant": "gpt"})

    def _fmt(batch):
        texts = []
        for convs, sys in zip(batch["conversations"], batch["system"]):
            msgs = [{"from": "system", "value": sys}] + convs
            texts.append(tokenizer.apply_chat_template(
                msgs, tokenize=False, add_generation_prompt=False))
        return {"text": texts}

    train_ds = Dataset.from_list(_train_snapshot).map(_fmt, batched=True)
    val_ds   = Dataset.from_list(_val_snapshot).map(_fmt,   batched=True)

    trainer = SFTTrainer(
        model=model, tokenizer=tokenizer,
        train_dataset=train_ds, eval_dataset=val_ds,
        dataset_text_field="text", max_seq_length=MAX_SEQ_LENGTH, packing=False,
        args=SFTConfig(
            per_device_train_batch_size = BATCH_SIZE,
            gradient_accumulation_steps = max(1, GRAD_ACCUMULATION // N_GPUS),
            num_train_epochs            = NUM_EPOCHS,
            learning_rate               = LEARNING_RATE,
            fp16                        = not torch.cuda.is_bf16_supported(),
            bf16                        = torch.cuda.is_bf16_supported(),
            logging_steps               = 20,
            eval_strategy               = "epoch",
            save_strategy               = "epoch",
            save_total_limit            = 2,
            load_best_model_at_end      = True,
            optim                       = "adamw_8bit",
            weight_decay                = 0.01,
            lr_scheduler_type           = "cosine",
            warmup_ratio                = 0.05,
            seed                        = 3407,
            output_dir                  = OUTPUT_DIR,
            report_to                   = "none",
            ddp_find_unused_parameters  = False,   # required for DDP
        ),
    )

    stats = trainer.train()

    # Only rank-0 saves and prints
    if int(os.environ.get("LOCAL_RANK", 0)) == 0:
        model.save_pretrained(OUTPUT_DIR)
        tokenizer.save_pretrained(OUTPUT_DIR)
        print(f"\nLoss : {stats.metrics.get('train_loss', 'N/A'):.4f}")
        print(f"Time : {stats.metrics['train_runtime']/60:.1f} min")
        print(f"VRAM : {torch.cuda.max_memory_reserved()/1024**3:.2f} GB")

# ── Launch ─────────────────────────────────────────────────────────────────────
from accelerate import notebook_launcher
notebook_launcher(train_fn, num_processes=N_GPUS)

## 8. Save LoRA Adapter to Drive

In [ ]:
import shutil

LOCAL_SAVE = OUTPUT_DIR
DRIVE_SAVE = "/content/drive/MyDrive/llama3_taiwan_hakka_translation_lora"

model.save_pretrained(LOCAL_SAVE)
tokenizer.save_pretrained(LOCAL_SAVE)
print(f"Saved locally : {LOCAL_SAVE}/")

if os.path.exists("/content/drive/MyDrive"):
    if os.path.exists(DRIVE_SAVE):
        shutil.rmtree(DRIVE_SAVE)
    shutil.copytree(LOCAL_SAVE, DRIVE_SAVE)
    print(f"Copied to Drive: {DRIVE_SAVE}/")

## 9. Evaluation — BLEU + BERTScore

Evaluates **both translation directions** on the sentence-level validation set.

Metrics:
- **Sentence BLEU** — character n-gram precision per sentence
- **Corpus BLEU** — corpus-level BLEU (standard MT metric)
- **BERTScore** — semantic similarity via `bert-base-chinese`

In [ ]:
# ── Pure-Python BLEU (no extra deps) ─────────────────────────────────────────
import math
from collections import Counter
from typing import List, Sequence

def _norm(text): return "".join((text or "").strip().split())
def _tok(text):  return list(_norm(text))

def _ngrams(tokens: Sequence[str], n: int) -> Counter:
    if len(tokens) < n: return Counter()
    return Counter(tuple(tokens[i:i+n]) for i in range(len(tokens)-n+1))

def sentence_bleu(reference: str, hypothesis: str, max_n: int = 4, smooth: float = 1.0) -> float:
    ref, hyp = _tok(reference), _tok(hypothesis)
    if not hyp: return 0.0
    precs = []
    for n in range(1, max_n+1):
        rc, hc = _ngrams(ref, n), _ngrams(hyp, n)
        clip = total = 0
        for ng, cnt in hc.items():
            clip  += min(cnt, rc.get(ng, 0))
            total += cnt
        precs.append((clip+smooth)/(total+smooth) if total else 0.0)
    if min(precs) <= 0: return 0.0
    geo = math.exp(sum(math.log(p) for p in precs) / max_n)
    bp  = 1.0 if len(hyp) >= len(ref) else math.exp(1.0 - len(ref)/max(len(hyp),1))
    return bp * geo

def corpus_bleu(references: List[str], hypotheses: List[str], max_n: int = 4) -> float:
    if len(references) != len(hypotheses) or not references: return 0.0
    clip_tot = [0]*max_n; hyp_tot = [0]*max_n
    ref_len = hyp_len = 0
    for ref, hyp in zip(references, hypotheses):
        rt, ht = _tok(ref), _tok(hyp)
        ref_len += len(rt); hyp_len += len(ht)
        for n in range(1, max_n+1):
            rc, hc = _ngrams(rt, n), _ngrams(ht, n)
            clip = total = 0
            for ng, cnt in hc.items():
                clip  += min(cnt, rc.get(ng, 0))
                total += cnt
            clip_tot[n-1] += clip; hyp_tot[n-1] += total
    precs = [(c/h if h else 0.0) for c, h in zip(clip_tot, hyp_tot)]
    if min(precs) <= 0 or hyp_len == 0: return 0.0
    geo = math.exp(sum(math.log(p) for p in precs) / max_n)
    bp  = 1.0 if hyp_len >= ref_len else math.exp(1.0 - ref_len/hyp_len)
    return bp * geo

print("BLEU functions ready.")

In [ ]:
# ── Load fine-tuned model for evaluation ─────────────────────────────────────
# Uses FastLanguageModel (same session — no reload needed after training)
FastLanguageModel.for_inference(model)

EVAL_MAX_NEW_TOKENS = 256
EVAL_BATCH_SIZE     = 8
EVAL_DO_SAMPLE      = False   # greedy for reproducibility
EVAL_REP_PENALTY    = 1.1

# Map evaluation mode → system prompt key
EVAL_MODE_SYSTEM = {
    "hakka2zh": "h2m",
    "zh2hakka": "m2h_sixian",
}


def clean_prediction(text: str) -> str:
    """Strip Llama-3 special tokens and keep first non-empty line."""
    text = re.split(r"<\|eot_id\|>|<\|im_end\|>", text)[0]
    text = re.sub(r"<\|[^|]+\|>", "", text).replace("\ufffd", "").strip()
    for line in text.splitlines():
        line = line.strip()
        if line:
            return line
    return ""


def build_eval_prompt(source_text: str, mode: str) -> str:
    messages = [
        {"from": "system", "value": SYSTEM[EVAL_MODE_SYSTEM[mode]]},
        {"from": "human",  "value": source_text},
    ]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )


def generate_batch(source_texts: List[str], mode: str) -> List[str]:
    prompts = [build_eval_prompt(s, mode) for s in source_texts]
    inputs  = tokenizer(prompts, return_tensors="pt", padding=True, truncation=False)
    inputs  = {k: v.to(model.device) for k, v in inputs.items()}
    with torch.no_grad():
        out_ids = model.generate(
            **inputs,
            max_new_tokens     = EVAL_MAX_NEW_TOKENS,
            do_sample          = EVAL_DO_SAMPLE,
            repetition_penalty = EVAL_REP_PENALTY,
            eos_token_id       = tokenizer.eos_token_id,
            pad_token_id       = tokenizer.pad_token_id,
        )
    results = []
    for i, oid in enumerate(out_ids):
        prompt_len = inputs["input_ids"][i].shape[0]
        raw = tokenizer.decode(oid[prompt_len:], skip_special_tokens=False)
        results.append(clean_prediction(raw))
    return results

print("Evaluation helpers ready.")

In [ ]:
# ── Run evaluation on validation set ─────────────────────────────────────────
# Val set = sentence pairs only (TSV + dict), suitable for BLEU
EVAL_LIMIT = None   # set an int (e.g. 50) to speed up during debugging

eval_results = {}

for mode in ["hakka2zh", "zh2hakka"]:
    direction_filter = "h2m" if mode == "hakka2zh" else "m2h"
    subset = [ex for ex in sent_val if ex.get("direction") == direction_filter]
    if EVAL_LIMIT:
        subset = subset[:EVAL_LIMIT]

    print(f"\n{'─'*55}")
    print(f"[{mode}]  {len(subset)} examples")
    print(f"{'─'*55}")

    rows     = []
    refs_all = []
    hyps_all = []

    for start in range(0, len(subset), EVAL_BATCH_SIZE):
        batch   = subset[start: start + EVAL_BATCH_SIZE]
        sources = [ex["hakka"]    if mode == "hakka2zh" else ex["mandarin"] for ex in batch]
        refs    = [ex["mandarin"] if mode == "hakka2zh" else ex["hakka"]    for ex in batch]

        try:
            preds = generate_batch(sources, mode)
        except Exception as e:
            print(f"  batch {start} failed: {e}")
            preds = [""] * len(batch)

        for src, ref, pred in zip(sources, refs, preds):
            if not pred:
                continue
            bleu = sentence_bleu(ref, pred)
            refs_all.append(ref)
            hyps_all.append(pred)
            rows.append({"source": src, "reference": ref,
                          "prediction": pred, "sentence_bleu": round(bleu, 6)})
            idx = len(rows)
            if idx <= 5 or idx % 50 == 0:
                print(f"  [{idx:3d}/{len(subset)}] BLEU={bleu:.4f}")
                print(f"         REF : {ref[:60]}")
                print(f"         PRED: {pred[:60]}")

    avg_bleu  = sum(r["sentence_bleu"] for r in rows) / max(len(rows), 1)
    corp_bleu = corpus_bleu(refs_all, hyps_all)

    eval_results[mode] = {
        "summary": {
            "samples":           len(rows),
            "avg_sentence_bleu": round(avg_bleu,  6),
            "corpus_bleu":       round(corp_bleu, 6),
        },
        "rows":      rows,
        "refs_all":  refs_all,
        "hyps_all":  hyps_all,
    }
    print(f"\n  avg BLEU   = {avg_bleu:.4f}")
    print(f"  corpus BLEU= {corp_bleu:.4f}")

In [ ]:
# ── BERTScore (bert-base-chinese) ─────────────────────────────────────────────
from bert_score import score as bert_score

print("Computing BERTScore…")
for mode, result in eval_results.items():
    refs  = result["refs_all"]
    hyps  = result["hyps_all"]
    if not refs:
        continue
    P, R, F1 = bert_score(
        hyps, refs,
        model_type = "bert-base-chinese",
        lang       = "zh",
        verbose    = False,
    )
    f1_list = F1.tolist()
    for row, f1 in zip(result["rows"], f1_list):
        row["bert_score_f1"] = round(f1, 6)
    avg_f1 = sum(f1_list) / len(f1_list)
    result["summary"]["bert_score_f1"] = round(avg_f1, 6)
    print(f"  [{mode}]  BERTScore F1 = {avg_f1:.4f}")

# Final summary table
print("\n" + "═"*60)
print("Evaluation Summary")
print("═"*60)
print(f"{'Direction':<12} {'Samples':>7} {'avg BLEU':>10} {'Corpus BLEU':>12} {'BERTScore F1':>13}")
print("-"*60)
for mode, result in eval_results.items():
    s = result["summary"]
    bert = f"{s.get('bert_score_f1', 0):.4f}"
    print(f"{mode:<12} {s['samples']:>7} {s['avg_sentence_bleu']:>10.4f} "
          f"{s['corpus_bleu']:>12.4f} {bert:>13}")

In [ ]:
# ── Save JSON report to Drive ─────────────────────────────────────────────────
import json as _json

# Remove large list fields before saving (keep only summary + first 100 rows)
report = {}
for mode, result in eval_results.items():
    report[mode] = {
        "summary": result["summary"],
        "samples": result["rows"][:100],
    }

REPORT_PATH = "/content/hakka_translation_eval_report.json"
with open(REPORT_PATH, "w", encoding="utf-8") as f:
    _json.dump(report, f, ensure_ascii=False, indent=2)
print(f"Report saved → {REPORT_PATH}")

# Copy to Drive
DRIVE_REPORT = "/content/drive/MyDrive/hakka_translation_eval_report.json"
if os.path.exists("/content/drive/MyDrive"):
    shutil.copy(REPORT_PATH, DRIVE_REPORT)
    print(f"Copied to Drive → {DRIVE_REPORT}")

## 10. Inference Tests

Test both translation directions and Hakka dialogue mode.

In [ ]:
from transformers import TextStreamer

FastLanguageModel.for_inference(model)

def translate(source_text: str, mode: str, max_new_tokens: int = 128) -> str:
    """Translate a single sentence; mode = 'hakka2zh' or 'zh2hakka'."""
    messages = [
        {"from": "system", "value": SYSTEM[EVAL_MODE_SYSTEM[mode]]},
        {"from": "human",  "value": source_text},
    ]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to("cuda")
    streamer = TextStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)
    model.generate(input_ids=inputs, streamer=streamer,
                   max_new_tokens=max_new_tokens, use_cache=True,
                   do_sample=False, repetition_penalty=1.1)

def chat_hakka(message: str, history: list = None, max_new_tokens: int = 200):
    """Hakka doctor–patient dialogue."""
    messages = [{"from": "system", "value": SYSTEM["dialogue"]}]
    if history:
        messages += history
    messages.append({"from": "human", "value": message})
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to("cuda")
    streamer = TextStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)
    model.generate(input_ids=inputs, streamer=streamer,
                   max_new_tokens=max_new_tokens, use_cache=True,
                   temperature=0.7, do_sample=True)

def test(n, label, fn, *args, **kwargs):
    print(f"\n{'='*60}")
    print(f"Test {n} — {label}")
    print(f"{'='*60}")
    fn(*args, **kwargs)
    print()

print("Inference helpers ready.")

In [ ]:
# Test 1 — Hakka → Mandarin (medical)
test(1, "Hakka → Mandarin (medical)", translate,
     "先生，𠊎這駁仔密密愛屙尿，日時頭差毋多半點鐘就愛走一到便所。",
     mode="hakka2zh")

# Test 2 — Mandarin → Hakka (medical)
test(2, "Mandarin → Hakka (medical)", translate,
     "醫生，我最近頻尿得很厲害，白天幾乎每半小時就要跑一次廁所。",
     mode="zh2hakka")

# Test 3 — Hakka → Mandarin (general, from dictionary domain)
test(3, "Hakka → Mandarin (general)", translate,
     "好朋友出國讀書幾年了，毋知佢這下好無？",
     mode="hakka2zh")

# Test 4 — Mandarin → Hakka (general)
test(4, "Mandarin → Hakka (general)", translate,
     "這樣的情況持續多久了？除了頻尿之外，有沒有尿急或尿痛？",
     mode="zh2hakka")

# Test 5 — Hakka dialogue (multi-turn)
test(5, "Hakka Dialogue (multi-turn)", chat_hakka,
     "主要係右膝，行樓梯个時候特別痛，休息一下會好一下。",
     history=[
         {"from": "human", "value": "先生，𠊎膝蓋仔痛好幾個禮拜了。"},
         {"from": "gpt",   "value": "恁樣哦，係哪位膝蓋仔？痛係持續个還係一陣一陣个？"},
     ])

## 11. Worst / Best Predictions Analysis

In [ ]:
for mode, result in eval_results.items():
    rows = result["rows"]
    if not rows:
        continue
    sorted_rows = sorted(rows, key=lambda r: r["sentence_bleu"])
    print(f"\n{'═'*55}")
    print(f"[{mode}] — 3 Worst predictions (lowest BLEU)")
    print("═"*55)
    for r in sorted_rows[:3]:
        print(f"  BLEU={r['sentence_bleu']:.4f}")
        print(f"  SRC : {r['source'][:70]}")
        print(f"  REF : {r['reference'][:70]}")
        print(f"  PRED: {r['prediction'][:70]}")
        print()

    print(f"[{mode}] — 3 Best predictions (highest BLEU)")
    print("─"*55)
    for r in sorted_rows[-3:][::-1]:
        print(f"  BLEU={r['sentence_bleu']:.4f}")
        print(f"  SRC : {r['source'][:70]}")
        print(f"  REF : {r['reference'][:70]}")
        print(f"  PRED: {r['prediction'][:70]}")
        print()

## 12. Gradio UI — Translation + Dialogue

Three tabs: Hakka→Mandarin · Mandarin→Hakka · Hakka Doctor Dialogue

In [ ]:
%%capture
!pip install gradio -q

In [ ]:
import gradio as gr

FastLanguageModel.for_inference(model)


def _generate(messages: list, max_new_tokens: int = 256,
               do_sample: bool = False, temperature: float = 0.7) -> str:
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to("cuda")
    with torch.no_grad():
        out_ids = model.generate(
            input_ids=inputs, max_new_tokens=max_new_tokens,
            use_cache=True, do_sample=do_sample,
            temperature=temperature if do_sample else 1.0,
            repetition_penalty=1.1,
        )
    return clean_prediction(
        tokenizer.decode(out_ids[0][inputs.shape[1]:], skip_special_tokens=False)
    )


def hakka_to_zh(text: str) -> str:
    if not text.strip(): return ""
    return _generate([{"from": "system", "value": SYSTEM["h2m"]},
                       {"from": "human",  "value": text}], max_new_tokens=128)


def zh_to_hakka(text: str, dialect: str) -> str:
    if not text.strip(): return ""
    key = "m2h_sixian" if dialect == "四縣腔" else "m2h_hailu"
    return _generate([{"from": "system", "value": SYSTEM[key]},
                       {"from": "human",  "value": text}], max_new_tokens=128)


def hakka_chat(message: str, history: list) -> str:
    conv = []
    for user_msg, bot_msg in history:
        conv.append({"from": "human", "value": user_msg})
        if bot_msg: conv.append({"from": "gpt", "value": bot_msg})
    return _generate([{"from": "system", "value": SYSTEM["dialogue"]}]
                     + conv + [{"from": "human", "value": message}],
                     do_sample=True, temperature=0.7)


with gr.Blocks(theme=gr.themes.Soft(), title="客家語翻譯助手") as demo:
    gr.Markdown("# 🌿 客家語翻譯助手（Llama-3-Taiwan 微調）")

    with gr.Tab("客語 → 繁中"):
        with gr.Row():
            inp_h2m = gr.Textbox(label="客語輸入", lines=4,
                                  placeholder="輸入客語漢字…")
            out_h2m = gr.Textbox(label="繁體中文輸出", lines=4, interactive=False)
        gr.Button("翻譯 →", variant="primary").click(hakka_to_zh, inp_h2m, out_h2m)
        gr.Examples([["先生，𠊎這駁仔密密愛屙尿，日時頭差毋多半點鐘就愛走一到便所。"],
                      ["好朋友出國讀書幾年了，毋知佢這下好無？"]], inputs=inp_h2m)

    with gr.Tab("繁中 → 客語"):
        with gr.Row():
            inp_m2h = gr.Textbox(label="繁體中文輸入", lines=4,
                                  placeholder="輸入繁體中文…")
            out_m2h = gr.Textbox(label="客語漢字輸出", lines=4, interactive=False)
        dialect = gr.Radio(["四縣腔", "海陸腔"], value="四縣腔", label="腔調")
        gr.Button("翻譯 →", variant="primary").click(zh_to_hakka, [inp_m2h, dialect], out_m2h)
        gr.Examples([["醫生，我最近頻尿得很厲害，白天幾乎每半小時就要跑一次廁所。"],
                      ["這樣的情況持續多久了？除了頻尿之外，有沒有尿急或尿痛？"]], inputs=inp_m2h)

    with gr.Tab("🏥 客語問診"):
        gr.ChatInterface(
            fn=hakka_chat,
            description="請用客語或普通話描述症狀，醫師將以客語回應。",
            examples=[["先生，𠊎膝蓋仔痛好幾個禮拜了。"],
                       ["𠊎腰背痛好久了，坐久了特別嚴重。"]],
            chatbot=gr.Chatbot(height=420),
            retry_btn="🔄", undo_btn="↩️", clear_btn="🗑️",
        )

demo.launch(share=True)

---
## Notes

| Item | Detail |
|------|--------|
| Data (sentences) | ~25 k pairs — TSV medical (294) + dict-hakka (~12 k) × 2 directions |
| Data (words) | ~100 k pairs — HAT Lexicon Sixian + Hailu × 2 directions |
| Training formats | H→M translation · M→H translation · Hakka dialogue |
| Dialect | 四縣腔 (Sixian) primary; 海陸腔 (Hailu) in lexicon word pairs |
| Evaluation | Sentence BLEU · Corpus BLEU · BERTScore (bert-base-chinese) |
| Val set | Sentence pairs only (not lexicon word pairs) |
| Next steps | Add more data; train with 海陸腔-only data; try larger LoRA rank |
